# BirdCLEF+ 2025 — SED model training (Sudhishna)

**The problem.** A label covers a whole recording (about 20 s), but we must predict every 5 s.
Jamie's improved baseline (v2) gives one score per 5 s and cannot say *where* a call happens.

**The idea.** A sound event detection (SED) model reads a longer chunk and scores **every moment**
for every species. During training, those moment-by-moment scores are combined into one score for
the chunk, which is compared with the recording's label. The model therefore learns where calls
are without ever being told. At test time we read the moment-by-moment scores for each 5 s window.

**What stays the same as v2**, so the comparison is fair: EfficientNet-B0, random crops, soundscape
noise + mixup, secondary labels at 0.5, AdamW with a one-cycle schedule, batch 64, 15 epochs, and
the same 80/20 split (seed 42). All runs except `sed20-hires` also use Jamie's spectrograms.

**What changes:** the input is 20 s instead of 5 s, and an SED head replaces the classifier.

**Sources.** The approach (SED on 20 s chunks, GeM frequency pooling) follows the description in
the 1st-place BirdCLEF+ 2025 write-up by N. Babych, which credits its SED head to the 4th-place
BirdCLEF 2021 write-up by tattaka (also the source of the chunk + peak loss). Attention pooling
for weak labels follows the PANNs paper (Kong et al. 2020). This code was written from those
descriptions; no other participant's code was used.

> Edit this master copy only, then run `python3 scripts/sudhishna_experiments.py` to refresh the
> per-experiment folders.

In [ ]:
EXPERIMENT = "sed20-last"   # smoke | sed20 | sed20-last | sed20-hires | sed5

## Settings
* `smoke`: a very short test that checks everything runs.
* `sed20`: the first full model. Saves the epoch with the highest `auc_peak`.
* `sed20-last`: the same, but saves the final epoch, after the learning rate has fallen to almost zero.
* `sed20-hires`: `sed20-last` on spectrograms with the settings in the 1st-place write-up
  (224 bands, n_fft 4096, hop 1252, 0–16 kHz), made by `sc4000-birdclef-m2-make-mels-224`.
* `sed5`: `sed20-last` on 5 s chunks (Jamie's spectrograms). It shows whether the 20 s chunk
  or the SED head is what keeps SED below v2, since v2 is also trained on 5 s.

In [ ]:
V2_RECIPE = dict(random_crop=True, noise=True, mixup=True, secondary=True)
WRITEUP_MELS = dict(n_fft=4096, hop_length=1252, n_mels=224, fmin=0, fmax=16_000)

SWITCHES = {
    "smoke": dict(V2_RECIPE, chunk_sec=20, epochs=1, max_steps=30),
    "sed20": dict(V2_RECIPE, chunk_sec=20),
    "sed20-last": dict(V2_RECIPE, chunk_sec=20, keep="last"),
    "sed20-hires": dict(V2_RECIPE, **WRITEUP_MELS, chunk_sec=20, keep="last"),
    "sed5": dict(V2_RECIPE, chunk_sec=5, keep="last"),
}

CFG = dict(
    # Spectrogram settings: must match make_mels and the inference notebook
    sr=32_000, n_fft=2048, hop_length=512, n_mels=128, fmin=50, fmax=14_000,
    chunk_sec=20, window_sec=5,
    # Training recipe: same as v2
    model="efficientnet_b0", epochs=15, batch_size=64, lr=1e-3, weight_decay=1e-2,
    val_frac=0.2, seed=42,
    keep="best",      # which weights to save: "best" (highest auc_peak) or "last" (final epoch)
    random_crop=False,
    noise=False, noise_prob=0.5, noise_db=(-20, -3),
    mixup=False, mixup_prob=0.5, mixup_alpha=0.4,
    secondary=False, secondary_weight=0.5,
    # SED head
    head_dim=512, dropout=0.25, gem_p=3.0, peak_loss_weight=0.5,
    # Above 0, stop each pass early (used by the short test only)
    max_steps=0,
)
CFG.update(SWITCHES[EXPERIMENT])
CFG["experiment"] = EXPERIMENT
print(EXPERIMENT, "->", {k: CFG[k] for k in ("chunk_sec", "epochs", "keep", "n_mels", "hop_length", "random_crop",
                                             "noise", "mixup", "secondary", "max_steps")})

In [ ]:
import os, ast, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG["seed"])


INPUT = Path("/kaggle/input")
SKIP = {"train_audio", "train_soundscapes", "test_soundscapes"}

def find_dir(marker, max_depth=4):
    """Find the folder holding `marker`, without walking through the 38,000 audio files."""
    for folder, subdirs, files in os.walk(INPUT):
        depth = len(Path(folder).relative_to(INPUT).parts)
        subdirs[:] = [d for d in subdirs if d not in SKIP] if depth < max_depth else []
        if marker in files:
            return Path(folder)
    raise FileNotFoundError(f"{marker} not found under {INPUT}")

t0 = time.time()
DATA, MELS = find_dir("sample_submission.csv"), find_dir("train_mels.npy")
OUT = Path("."); FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Data: {DATA} | mels: {MELS} | found in {time.time() - t0:.1f}s")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
assert DEVICE == "cuda", "No GPU: check the notebook settings and that the phone is verified"

## Load the spectrograms
These are whole-clip spectrograms (up to 30 s per clip), made once by a separate notebook, so nothing
has to be recomputed. With Jamie's settings, 20 s of audio is 1,251 frames; with the write-up's, it is 512.

In [ ]:
t0 = time.time()
X = np.load(MELS / "train_mels.npy")                 # (total_frames, n_mels), uint8
index = pd.read_csv(MELS / "train_index.csv")
NOISE = np.load(MELS / "noise_mels.npy") if CFG["noise"] else None
mcfg = json.load(open(MELS / "mels_cfg.json"))
for k in ("sr", "n_fft", "hop_length", "n_mels", "fmin", "fmax"):
    assert mcfg[k] == CFG[k], f"spectrogram setting {k} differs from make_mels"

W = 1 + CFG["chunk_sec"] * CFG["sr"] // CFG["hop_length"]     # frames per chunk
STORE_DB, TOP_DB = float(mcfg["store_db"]), 80.0
START, NFRAMES = index["start"].values, index["n_frames"].values
print(f"mels {X.shape} {X.nbytes / 1e9:.1f} GB | {len(index)} clips | chunk {W} frames | "
      f"shorter than the chunk: {(NFRAMES < W).mean():.1%} | {time.time() - t0:.0f}s")

## Labels and validation split
Same species order and same 80/20 split as Jamie's runs. Secondary species get a soft target of 0.5.

In [ ]:
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labels = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labels = []
    return [sp2idx[l] for l in labels if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va))

## Dataset: cutting a chunk
* **Longer clips:** a random 20 s piece during training, the first 20 s during validation.
* **Shorter clips (about half):** placed at a random position inside the 20 s and surrounded by
  silence, so the model cannot assume calls always start at the beginning.
* **Noise:** with probability 0.5, a 20 s piece of an unlabelled soundscape is added on the power
  scale, exactly as in Jamie's v2. It also fills the silent padding with realistic background.
* **Scaling:** each chunk is scaled by its own loudest point, exactly as the inference notebook does.

In [ ]:
def to_db(u8):
    """uint8 from make_mels -> dB relative to the clip's loudest point. Returns (n_mels, frames)."""
    return u8.T.astype(np.float32) / 255.0 * STORE_DB - STORE_DB

def to_input(db):
    """0 dB = this chunk's loudest point, keep 80 dB below it, scale to [0, 1]."""
    return ((np.maximum(db - db.max(), -TOP_DB) + TOP_DB) / TOP_DB).astype(np.float32)

class ChunkDataset(torch.utils.data.Dataset):
    def __init__(self, idx, train):
        self.idx, self.train = idx, train

    def __len__(self):
        return len(self.idx)

    def cut_chunk(self, j):
        start, n = START[j], NFRAMES[j]
        randomise = self.train and CFG["random_crop"]
        if n >= W:
            offset = np.random.randint(0, n - W + 1) if randomise else 0
            return to_db(X[start + offset : start + offset + W])
        db = np.full((CFG["n_mels"], W), -STORE_DB, dtype=np.float32)
        position = np.random.randint(0, W - n + 1) if randomise else 0
        db[:, position : position + n] = to_db(X[start : start + n])
        return db

    def __getitem__(self, i):
        j = self.idx[i]
        db = self.cut_chunk(j)
        if self.train and CFG["noise"] and np.random.rand() < CFG["noise_prob"]:
            k = np.random.randint(len(NOISE))
            offset = np.random.randint(0, NOISE.shape[1] - W + 1)
            noise = to_db(NOISE[k, offset : offset + W])
            gain_db = np.random.uniform(*CFG["noise_db"])
            power = 10.0 ** ((db - db.max()) / 10.0) + 10.0 ** ((noise - noise.max() + gain_db) / 10.0)
            db = 10.0 * np.log10(power)
        target = np.zeros(len(species), dtype=np.float32)
        if self.train and CFG["secondary"]:
            target[sec_idx[j]] = CFG["secondary_weight"]
        target[y_idx[j]] = 1.0          # validation uses the primary label only, like Jamie's runs
        return torch.from_numpy(to_input(db))[None], torch.from_numpy(target)

NW = min(4, os.cpu_count())
dl_tr = torch.utils.data.DataLoader(ChunkDataset(idx_tr, True), batch_size=CFG["batch_size"],
                                    shuffle=True, num_workers=NW, drop_last=True,
                                    persistent_workers=True)
dl_va = torch.utils.data.DataLoader(ChunkDataset(idx_va, False), batch_size=CFG["batch_size"] * 2,
                                    shuffle=False, num_workers=NW)

## The SED model
1. **Backbone:** EfficientNet-B0 turns the spectrogram into features, shrinking it 32 times. It
   normally averages everything into one score; we keep the time axis instead. 20 s becomes 40 time
   steps of 0.5 s each with Jamie's spectrograms, or 16 time steps of 1.25 s each with the write-up's.
2. **Frequency pooling (GeM):** collapses the frequency axis while favouring strong activations.
3. **Moment-by-moment scores:** one score per species per time step. This is what we use at test time.
4. **Attention:** learns which moments matter for each species, giving one score for the whole chunk.

**Loss.** We compare two scores with the label and average them:
* the **attention score** for the whole chunk, which handles calls spread across several moments;
* the **peak score**, the highest moment for each species. It pushes every moment down for absent
  species, and at least one moment up for present ones, which keeps the moment-by-moment scores
  reliable. The 4th-place BirdCLEF 2021 write-up (tattaka) also adds this peak term to the chunk
  loss, with weights 1 and 0.5; we use 0.5 and 0.5 and did not tune them.

In [ ]:
class SEDModel(nn.Module):
    """EfficientNet that keeps the time axis, so it can score every moment of the chunk."""

    def __init__(self, cfg, n_classes, pretrained):
        super().__init__()
        self.backbone = timm.create_model(cfg["model"], pretrained=pretrained, in_chans=1,
                                          num_classes=0, global_pool="")
        channels, hidden, drop = self.backbone.num_features, cfg["head_dim"], cfg["dropout"]
        self.gem_p = cfg["gem_p"]
        self.frame_net = nn.Sequential(nn.Dropout(drop), nn.Conv1d(channels, hidden, 1),
                                       nn.ReLU(inplace=True), nn.Dropout(drop))
        self.species_scores = nn.Conv1d(hidden, n_classes, 1)
        self.attention = nn.Conv1d(hidden, n_classes, 1)
        nn.init.constant_(self.species_scores.bias, -4.0)  # start with "probably not calling"

    def pool_frequency(self, features):
        """GeM pooling: (batch, channels, freq, time) -> (batch, channels, time)."""
        positive = features.float().clamp(min=1e-4)
        return positive.pow(self.gem_p).mean(dim=2).pow(1.0 / self.gem_p)

    def forward(self, mel):
        features = self.backbone.forward_features(mel)
        hidden = self.frame_net(self.pool_frequency(features))
        frame_logits = self.species_scores(hidden)                 # (batch, species, time)
        weights = torch.softmax(self.attention(hidden), dim=-1)    # which moments matter
        clip_logits = (weights * frame_logits).sum(dim=-1)         # attention pooling
        peak_logits = frame_logits.max(dim=-1).values              # loudest moment per species
        return frame_logits, clip_logits, peak_logits

In [ ]:
model = SEDModel(CFG, len(species), pretrained=True).to(DEVICE)

with torch.no_grad():
    frame_logits, clip_logits, peak_logits = model(torch.zeros(2, 1, CFG["n_mels"], W, device=DEVICE))
T = frame_logits.shape[-1]
assert frame_logits.shape[:2] == (2, len(species)) and clip_logits.shape == (2, len(species))
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters | {W} input frames -> "
      f"{T} time steps ({CFG['chunk_sec'] / T:.2f} s each)")

STEPS_PER_EPOCH = CFG["max_steps"] or len(dl_tr)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG["lr"], total_steps=CFG["epochs"] * STEPS_PER_EPOCH, pct_start=0.1)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
bce = nn.BCEWithLogitsLoss()

def sed_loss(clip_logits, peak_logits, target):
    w = CFG["peak_loss_weight"]
    return (1 - w) * bce(clip_logits.float(), target) + w * bce(peak_logits.float(), target)

def mixup(xb, yb):
    lam = float(np.random.beta(CFG["mixup_alpha"], CFG["mixup_alpha"]))
    perm = torch.randperm(xb.size(0), device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], torch.maximum(yb, yb[perm])

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro")

## Training loop
Validation uses clean hold-out clips and reports three scores:
* `auc_peak`: the highest moment in the chunk. With `keep="best"`, this picks the epoch to save.
* `auc_clip`: the attention score for the chunk.
* `auc_5s`: the highest moment in the first 5 s only. This is the closest comparison with Jamie's
  local scores. Local scores overstate the leaderboard, so the leaderboard remains the real test.

`keep="last"` exists because `sed20` saved epoch 7, when the learning rate was still 64% of its
peak, on an `auc_peak` lead of only 0.0003 over epoch 13.

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    clip_p, peak_p, first_p, targets, loss_sum = [], [], [], [], 0.0
    first_steps = max(1, round(T * CFG["window_sec"] / CFG["chunk_sec"]))
    for step, (xb, yb) in enumerate(dl_va):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            frame_logits, clip_logits, peak_logits = model(xb)
        loss_sum += sed_loss(clip_logits, peak_logits, yb).item() * len(xb)
        clip_p.append(torch.sigmoid(clip_logits.float()).cpu().numpy())
        peak_p.append(torch.sigmoid(peak_logits.float()).cpu().numpy())
        first = frame_logits[..., :first_steps].max(dim=-1).values
        first_p.append(torch.sigmoid(first.float()).cpu().numpy())
        targets.append(yb.cpu().numpy())
        if CFG["max_steps"] and step + 1 >= CFG["max_steps"]:
            break
    t = np.concatenate(targets)
    return (loss_sum / len(t), macro_auc(t, np.concatenate(peak_p)),
            macro_auc(t, np.concatenate(clip_p)), macro_auc(t, np.concatenate(first_p)))

history, best_auc, best_epoch, saved_epoch = [], 0.0, 0, 0
for epoch in range(1, CFG["epochs"] + 1):
    model.train(); t0 = time.time(); loss_sum, n_seen = 0.0, 0
    for step, (xb, yb) in enumerate(dl_tr):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        if CFG["mixup"] and np.random.rand() < CFG["mixup_prob"]:
            xb, yb = mixup(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            _, clip_logits, peak_logits = model(xb)
            loss = sed_loss(clip_logits, peak_logits, yb)
        assert torch.isfinite(loss), f"loss became {loss.item()} at epoch {epoch} step {step}"
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(xb); n_seen += len(xb)
        if CFG["max_steps"] and step + 1 >= CFG["max_steps"]:
            break
    train_sec = time.time() - t0
    va_loss, auc_peak, auc_clip, auc_5s = evaluate()
    history.append(dict(epoch=epoch, train_loss=loss_sum / n_seen, val_loss=va_loss, val_auc=auc_peak,
                        val_auc_clip=auc_clip, val_auc_5s=auc_5s))
    flag = ""
    improved = auc_peak > best_auc
    if improved:
        best_auc, best_epoch = auc_peak, epoch
    if improved if CFG["keep"] == "best" else epoch == CFG["epochs"]:
        torch.save(model.state_dict(), OUT / "model.pth"); saved_epoch = epoch; flag = "  <- saved"
    print(f"epoch {epoch:2d} | train {loss_sum / n_seen:.4f} | val {va_loss:.4f} | auc_peak {auc_peak:.4f} | "
          f"auc_clip {auc_clip:.4f} | auc_5s {auc_5s:.4f} | {train_sec:.0f}s train, "
          f"{time.time() - t0:.0f}s total{flag}", flush=True)

sec_per_step = train_sec / STEPS_PER_EPOCH
print(f"[{EXPERIMENT}] best auc_peak: {best_auc:.4f} (epoch {best_epoch}) | saved weights: epoch {saved_epoch}")
print(f"Training speed: {sec_per_step:.2f} s/step, about {sec_per_step * len(dl_tr) / 60:.1f} min per full "
      f"epoch and {sec_per_step * len(dl_tr) * 15 / 60:.0f} min of training for 15 epochs "
      f"(validation not included)")

## Save what the inference notebook needs, plus figures for the report
`frame_predictions` shows the point of the model: the spectrogram of a hold-out clip above the
moment-by-moment score for its labelled species. If SED works, the score should rise when the call
happens, even though training only ever had a label for the whole recording.

In [ ]:
cfg_out = dict(CFG, best_val_auc=best_auc, best_epoch=best_epoch, saved_epoch=saved_epoch,
               frames_per_chunk=W, time_steps=T)
cfg_out["noise_db"] = list(CFG["noise_db"])
json.dump(cfg_out, open(OUT / "cfg.json", "w"), indent=1)
json.dump(species, open(OUT / "species.json", "w"))
hist = pd.DataFrame(history); hist.to_csv(OUT / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="Loss")
hist.plot(x="epoch", y=["val_auc", "val_auc_clip", "val_auc_5s"], ax=axes[1],
          title=f"Validation macro ROC-AUC ({EXPERIMENT})")
plt.tight_layout(); plt.savefig(FIG / f"training_{EXPERIMENT}.png", dpi=200); plt.show()

model.load_state_dict(torch.load(OUT / "model.pth", map_location=DEVICE)); model.eval()
long_clips = [j for j in idx_va if NFRAMES[j] >= W]
picks = np.random.default_rng(0).choice(long_clips, min(3, len(long_clips)), replace=False)
fig, axes = plt.subplots(2 * len(picks), 1, figsize=(11, 2.2 * len(picks) * 2), sharex=True,
                         gridspec_kw=dict(height_ratios=[2, 1] * len(picks)))
for n, j in enumerate(picks):
    x = to_input(to_db(X[START[j] : START[j] + W]))
    with torch.no_grad():
        frames = model(torch.from_numpy(x)[None, None].to(DEVICE))[0][0, y_idx[j]]
    prob = torch.sigmoid(frames).cpu().numpy()
    seconds = (np.arange(T) + 0.5) * CFG["chunk_sec"] / T
    axes[2 * n].imshow(x, origin="lower", aspect="auto", cmap="magma", extent=[0, CFG["chunk_sec"], 0, CFG["n_mels"]])
    axes[2 * n].set_ylabel("Mel band"); axes[2 * n].set_title(f"Hold-out clip: {species[y_idx[j]]}")
    axes[2 * n + 1].plot(seconds, prob); axes[2 * n + 1].set_ylim(0, 1); axes[2 * n + 1].set_ylabel("Score")
axes[-1].set_xlabel("Seconds")
plt.tight_layout(); plt.savefig(FIG / f"frame_predictions_{EXPERIMENT}.png", dpi=200); plt.show()